In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage
from scipy import ndimage
import math

# Hydraulic resistance calculation

In [ ]:
filename = 'Leopard'
raw_dimension = 1000

In [ ]:
boundaries_pressure_x = np.load(f'../saved_dicts_and_lists//{filename}_halfthroat_boundaries_pressure_x.npy', allow_pickle=True).item()
boundaries_pressure_y = np.load(f'../saved_dicts_and_lists//{filename}_halfthroat_boundaries_pressure_y.npy', allow_pickle=True).item()
boundaries_pressure_z = np.load(f'../saved_dicts_and_lists//{filename}_halfthroat_boundaries_pressure_z.npy', allow_pickle=True).item()

In [ ]:
halfthroat_boundaries_numbers = np.load(f'../saved_dicts_and_lists/{filename}_halfthroat_boundaries_numbers.npy', allow_pickle=True).item()

In [ ]:
halfthroat_boundaries = {i[0]: [] for i in halfthroat_boundaries_numbers.keys()}
for i in halfthroat_boundaries_numbers.keys():
    halfthroat_boundaries[i[0]].append(halfthroat_boundaries_numbers[i])

## Via the flow between pores

### Mean pressure on the boundary

In [ ]:
flow_x = np.load(f'../saved_dicts_and_lists/{filename}_pores_flow_x.npy', allow_pickle=True).item()
flow_y = np.load(f'../saved_dicts_and_lists/{filename}_pores_flow_y.npy', allow_pickle=True).item()
flow_z = np.load(f'../saved_dicts_and_lists/{filename}_pores_flow_z.npy', allow_pickle=True).item()

In [ ]:
pores_boundaries_numbers = np.load(f'../saved_dicts_and_lists/{filename}_pores_boundaries_numbers.npy', allow_pickle=True).item()

In [ ]:
g_x1 = {}
pressure_x = {}
for i in flow_x.keys():
    if i[0] < i[1]:
        continue

    htnum = abs(pores_boundaries_numbers[i])
    flow = abs(flow_x[i] - flow_x[i[::-1]]) / 2
    
    if htnum not in halfthroat_boundaries or len(halfthroat_boundaries[htnum]) != 2:
        continue

    
    g_x1[htnum] = flow / abs(boundaries_pressure_x[halfthroat_boundaries[htnum][0]] - boundaries_pressure_x[halfthroat_boundaries[htnum][1]])
    pressure_x[htnum] = abs(boundaries_pressure_x[halfthroat_boundaries[htnum][0]] - boundaries_pressure_x[halfthroat_boundaries[htnum][1]])

g_y1 = {}
pressure_y = {}
for i in flow_y.keys():
    if i[0] < i[1]:
        continue

    htnum = abs(pores_boundaries_numbers[i])
    flow = abs(flow_y[i] - flow_y[i[::-1]]) / 2
    
    if htnum not in halfthroat_boundaries or len(halfthroat_boundaries[htnum]) != 2:
        continue
        
    g_y1[htnum] = flow / abs(boundaries_pressure_y[halfthroat_boundaries[htnum][0]] - boundaries_pressure_y[halfthroat_boundaries[htnum][1]])
    pressure_y[htnum] = abs(boundaries_pressure_y[halfthroat_boundaries[htnum][0]] - boundaries_pressure_y[halfthroat_boundaries[htnum][1]])

g_z1 = {}
pressure_z = {}
for i in flow_z.keys():
    if i[0] < i[1]:
        continue

    htnum = abs(pores_boundaries_numbers[i])
    flow = abs(flow_z[i] - flow_z[i[::-1]]) / 2
    
    if htnum not in halfthroat_boundaries or len(halfthroat_boundaries[htnum]) != 2:
        continue

        
    g_z1[htnum] = flow / abs(boundaries_pressure_z[halfthroat_boundaries[htnum][0]] - boundaries_pressure_z[halfthroat_boundaries[htnum][1]])
    pressure_z[htnum] = abs(boundaries_pressure_z[halfthroat_boundaries[htnum][0]] - boundaries_pressure_z[halfthroat_boundaries[htnum][1]])

## Final g calculation

In [ ]:
def volume(body):
    return len(np.where(body)[0])

volume_dict = {}

halfthroats_img = np.fromfile(f"../raw_data/{filename}_halfthroats.raw", dtype=np.int32, sep="")
halfthroats_img = halfthroats_img.reshape([raw_dimension, raw_dimension, raw_dimension])

objects = ndimage.find_objects(halfthroats_img)

for val in range(1, len(objects) + 1):
    if objects[val - 1] != None:
        try:
            pore = np.copy(halfthroats_img[objects[val - 1]])
            pore[pore != val] = False
            pore[pore == val] = True

            volume_dict[val] = volume(pore)
        except Exception as e:
            print(val, e)
    if val % 1000 == 0:
        print(f'current: {val}')

In [ ]:
g = {}
for i in g_x1.keys():
    try:
        g[i] = (g_x1[i] + g_y1[i] + g_z1[i]) / (3 * volume_dict[i])

    except Exception as e:
        print(e)

# Shape factor calculation

In [ ]:
def area(body):
    area = 0

    body1 = np.full((body.shape[0] + 2, body.shape[1] + 2, body.shape[2] + 2), False)
    body1[1:body1.shape[0] - 1, 1:body1.shape[1] - 1, 1:body1.shape[2] - 1] = body
    
    for i, j, k in np.transpose(np.where(
        skimage.segmentation.find_boundaries(body1, connectivity=1, mode='inner')
    )):
        directions = [
            (1, 0, 0), (-1, 0, 0),
            (0, 1, 0), (0, -1, 0),
            (0, 0, 1), (0, 0, -1)
        ]
        for dx, dy, dz in directions:
            ni, nj, nk = i + dx, j + dy, k + dz
            try:
                if not body1[ni, nj, nk]:
                    area += 1
            except Exception as e:
                print(e)

    return area

def volume(body):
    return len(np.where(body)[0])

def convex_hull(body):
    points = np.transpose(np.where(body))
    hull = scipy.spatial.ConvexHull(points)
    deln = scipy.spatial.Delaunay(points[hull.vertices])
    idx = np.stack(np.indices(body.shape), axis=-1)
    out_idx = np.nonzero(deln.find_simplex(idx) + 1)
    out_img = np.zeros(body.shape)
    out_img[out_idx] = 1
    return out_img.astype(bool)

In [ ]:
pores_boundaries_numbers = np.load(f'../saved_dicts_and_lists/{filename}_pores_boundaries_numbers.npy', allow_pickle=True).item()
numbers_pores_boundaries = {pores_boundaries_numbers[i]: i for i in pores_boundaries_numbers.keys()}

In [ ]:
halfthroats_img = np.fromfile(f"../raw_data/{filename}_halfthroats.raw", dtype=np.int32, sep="")
halfthroats_img = halfthroats_img.reshape([raw_dimension, raw_dimension, raw_dimension])

In [ ]:
objects = ndimage.find_objects(halfthroats_img)

## V. Postnikov's descriptor

In [ ]:
from julia.api import Julia
jl = Julia(compiled_modules=False)
from julia import Main

In [ ]:
Main.eval('include("../scripts/ShapeParameter.jl")')

In [ ]:
Main.using('.ShapeParameter')

! It turned out that the algorithm does not perform erosion from the boundary

In [ ]:
def calc_G_vp(body):
    body1 = np.full((body.shape[0] + 2, body.shape[1] + 2, body.shape[2] + 2), False)
    body1[1:body1.shape[0] - 1, 1:body1.shape[1] - 1, 1:body1.shape[2] - 1] = body

    k = 100 // min(body.shape) + 1

    return Main.ShapeParameter.parameter(np.kron(body1, np.ones((k, k, k))).astype('bool'))

In [ ]:
import time
timestamp = time.time()

In [ ]:
G_vp = {}
for val in range(1, len(objects) + 1):
    if objects[val - 1] != None:
        try:
            pore = np.copy(halfthroats_img[objects[val - 1]])
            pore[pore != val] = False
            pore[pore == val] = True
            
            G_vp[val] = calc_G_vp(pore)
        except Exception as e:
            print(val, e)
    if val % 100 == 0:
        print(f'current: {val}, time: {time.time() - timestamp}')
        timestamp = time.time()
    if val % 1000 == 0:
        np.save(f'../saved_dicts_and_lists/{filename}_G_vp.npy', G_vp)

np.save(f'../saved_dicts_and_lists/{filename}_G_vp.npy', G_vp)

### If interrupted, load the saved results

In [ ]:
G_vp = np.load(f'../saved_dicts_and_lists/{filename}_G_vp.npy', allow_pickle=True).item()
print(len(G_vp))

## Convex hull based descriptors

In [ ]:
import scipy

In [ ]:
G_1 = {}
G_2 = {}
G_3 = {}
volume_dict = {}
area_dict = {}
for val in range(1, len(objects) + 1):
    if objects[val - 1] != None:
        try:
            pore = np.copy(halfthroats_img[objects[val - 1]])
            pore[pore != val] = False
            pore[pore == val] = True

            hull = convex_hull(pore)
            
            G_1[val] = (volume(pore) ** 2 / area(pore) ** 3)  / 8.842e-3
            G_2[val] = area(hull) / area(pore)
            G_3[val] = volume(pore) / volume(hull)
            volume_dict[val] = volume(pore)
            area_dict[val] = area(pore)
        except Exception as e:
            print(val, e)
    if val % 1000 == 0:
        print(f'current: {val}')


## Angles and sizes along the principal axes

### Compute the inflow and outflow points as the mean of the edges

In [ ]:
halfthroats_boundaries_img = np.fromfile(f"../raw_data/{filename}_halfthroats_boundaries.raw", dtype=np.int32, sep="")
halfthroats_boundaries_img = halfthroats_boundaries_img.reshape([raw_dimension, raw_dimension, raw_dimension])

In [ ]:
x_coord = np.zeros((raw_dimension, raw_dimension, raw_dimension))
y_coord = np.zeros((raw_dimension, raw_dimension, raw_dimension))
z_coord = np.zeros((raw_dimension, raw_dimension, raw_dimension))
for i in range(raw_dimension):
    x_coord[i, :, :] = i
    y_coord[:, i, :] = i
    z_coord[:, :, i] = i

In [ ]:
index = np.unique(halfthroats_boundaries_img)[1:]

In [ ]:
val_indices = scipy.ndimage.value_indices(halfthroats_boundaries_img)

In [ ]:
x_boundaries = scipy.ndimage.sum_labels(x_coord, halfthroats_boundaries_img, index = index)
y_boundaries = scipy.ndimage.sum_labels(y_coord, halfthroats_boundaries_img, index = index)
z_boundaries = scipy.ndimage.sum_labels(z_coord, halfthroats_boundaries_img, index = index)

In [ ]:
halfthroat_boundaries_numbers = np.load(f'../saved_dicts_and_lists/{filename}_halfthroat_boundaries_numbers.npy', allow_pickle=True).item()

In [ ]:
halfthroat_boundaries = {i[0]: [] for i in halfthroat_boundaries_numbers.keys()}
for i in halfthroat_boundaries_numbers.keys():
    halfthroat_boundaries[i[0]].append(halfthroat_boundaries_numbers[i])

In [ ]:
borders_center = {}
for i in range(len(index)):
    borders_center[index[i]] = np.array([x_boundaries[i], y_boundaries[i], z_boundaries[i]]) / len(val_indices[index[i]][0])

In [ ]:
flow_direction_vector = {}
for i in halfthroat_boundaries.keys():
    if len(halfthroat_boundaries[i]) == 2:
        if halfthroat_boundaries[i][0] not in index or halfthroat_boundaries[i][1] not in index:
            print("how is it possible")
            continue
        flow_direction_vector[i] = borders_center[halfthroat_boundaries[i][0]] - borders_center[halfthroat_boundaries[i][1]]

In [ ]:
import numpy as np

def principal_axes_pca(voxel_grid):
    """
    Compute the principal axes of a 3D voxel body using PCA.
    
    Args:
        voxel_grid: 3D numpy array where 1 indicates the presence of a voxel.
        
    Returns:
        centroid: Center of mass of the voxel body (1D array of length 3).
        axes: Principal axes as a 3x3 matrix (each column is an eigenvector 
              in descending order of eigenvalue).
    """
    # Get coordinates of all active voxels
    coords = np.argwhere(voxel_grid)
    
    if len(coords) == 0:
        raise ValueError("The voxel grid is empty.")
    
    # Calculate centroid (center of mass)
    centroid = np.mean(coords, axis=0)
    
    # Center the coordinates
    centered_coords = coords - centroid
    
    # Compute covariance matrix (population covariance)
    cov_matrix = centered_coords.T @ centered_coords / len(coords)
    
    # Eigen decomposition; eigenvectors are columns of 'eigenvecs'
    eigenvalues, eigenvecs = np.linalg.eigh(cov_matrix)
    
    # Sort eigenvectors by descending eigenvalues
    sorted_indices = np.argsort(eigenvalues)[::-1]
    axes = eigenvecs[:, sorted_indices]
    
    return centroid, axes


In [ ]:
def compute_body_measures(voxels, u, v, w):
    # Normalize the basis vectors to unit length
    u_hat = u / np.linalg.norm(u)
    v_hat = v / np.linalg.norm(v)
    w_hat = w / np.linalg.norm(w)
    
    # Compute half-length for one voxel along each basis direction
    half_length_u = 0.5 * (np.abs(u_hat[0]) + np.abs(u_hat[1]) + np.abs(u_hat[2]))
    half_length_v = 0.5 * (np.abs(v_hat[0]) + np.abs(v_hat[1]) + np.abs(v_hat[2]))
    half_length_w = 0.5 * (np.abs(w_hat[0]) + np.abs(w_hat[1]) + np.abs(w_hat[2]))
    
    # Get coordinates of all body voxels
    points = np.argwhere(voxels)
    
    # If no body voxels, return zero extents
    if points.size == 0:
        return 0.0, 0.0, 0.0
    
    # Project voxel centers onto the normalized basis vectors
    proj_u = np.dot(points, u_hat)
    proj_v = np.dot(points, v_hat)
    proj_w = np.dot(points, w_hat)
    
    # Calculate extents along each basis direction
    min_u = np.min(proj_u) - half_length_u
    max_u = np.max(proj_u) + half_length_u
    extent_u = max_u - min_u
    
    min_v = np.min(proj_v) - half_length_v
    max_v = np.max(proj_v) + half_length_v
    extent_v = max_v - min_v
    
    min_w = np.min(proj_w) - half_length_w
    max_w = np.max(proj_w) + half_length_w
    extent_w = max_w - min_w
    
    return extent_u, extent_v, extent_w

### Border shapes

In [ ]:
border_1_G4 = {}
border_2_G4 = {}

for val in range(1, len(objects) + 1):
    if objects[val - 1] != None:
        try:
            border1 = np.copy(halfthroats_boundaries_img[objects[val - 1]])
            border1[border1 != halfthroat_boundaries[val][0]] = False
            border1[border1 == halfthroat_boundaries[val][0]] = True

            border2 = np.copy(halfthroats_boundaries_img[objects[val - 1]])
            border2[border2 != halfthroat_boundaries[val][1]] = False
            border2[border2 == halfthroat_boundaries[val][1]] = True

            centroid, axes = principal_axes_pca(border1)
            paxes = np.transpose(axes)
            
            lengths = compute_body_measures(border1, *paxes)
            border_1_G4[val] = lengths[2] / lengths[0]

            centroid, axes = principal_axes_pca(border2)
            paxes = np.transpose(axes)
            
            lengths = compute_body_measures(border2, *paxes)
            border_2_G4[val] = lengths[2] / lengths[0]
        except Exception as e:
            print(val, e)
    if val % 1000 == 0:
        print(f'current: {val}')

### Compute the relative area of the borders

In [ ]:
def area_voxels(body):
    struct = np.array(
        [[[0, 0, 0],
          [0, 1, 0],
          [0, 0, 0]],
         [[0, 1, 0],
          [1, 1, 1],
          [0, 1, 0]],
         [[0, 0, 0],
          [0, 1, 0],
          [0, 0, 0]]]
    )
    return len(np.where(np.logical_and(
        body,
        np.logical_not(ndimage.binary_erosion(body, structure=struct))
    ))[0])

In [ ]:
area_dict = {}
for val in range(1, len(objects) + 1):
    if objects[val - 1] != None:
        try:
            pore = np.copy(halfthroats_img[objects[val - 1]])
            pore[pore != val] = False
            pore[pore == val] = True

            area_dict[val] = area_voxels(pore)
        except:
            print(1)
    if val % 1000 == 0:
        print(val)

In [ ]:
relative_area1 = {}
relative_area2 = {}
for i in halfthroat_boundaries.keys():
    if len(halfthroat_boundaries[i]) == 2:
        if halfthroat_boundaries[i][0] not in index or halfthroat_boundaries[i][1] not in index:
            continue
        if i not in area_dict.keys():
            continue
        relative_area1[i] = len(val_indices[halfthroat_boundaries[i][0]][0]) / area_dict[i]
        relative_area2[i] = len(val_indices[halfthroat_boundaries[i][1]][0]) / area_dict[i]

### Angles

In [ ]:
def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

In [ ]:
def vector_length(vector):
    return sum(vector * vector) ** 0.5

In [ ]:
def vectors_angle(vec1, vec2):
    if np.clip(np.dot(unit_vector(vec1), unit_vector(vec2)), -1.0, 1.0) != np.dot(unit_vector(vec1), unit_vector(vec2)):
        raise Exception("vector_angle_wtf")
    return np.arccos(np.clip(np.dot(unit_vector(vec1), unit_vector(vec2)), -1.0, 1.0))

In [ ]:
import numpy as np

def find_distance_to_voxel(
    voxel_space: np.ndarray,
    start_point: np.ndarray,
    direction_vector: np.ndarray,
    radius: float,
    max_distance: float = 200.0,
    step_size: float = 0.10
) -> float | None:
    """
    Finds the distance from the start point to the nearest 'True' voxel
    along the given direction, taking the search radius into account.

    Args:
        voxel_space (np.ndarray): 3D numpy array of boolean values
                                  representing the voxel space.
        start_point (np.ndarray): Start point of the ray (numpy array with 3 elements).
        direction_vector (np.ndarray): Direction vector (numpy array with 3 elements).
        radius (float): Search radius around each point on the ray.
                        Voxels in this neighbourhood are checked.
        max_distance (float, optional): Maximum distance the ray
                                        advances. Default is 1000.0.
        step_size (float, optional): Step size when moving along the ray.
                                     Smaller values increase accuracy but
                                     decrease performance. Default is 0.5.

    Returns:
        float | None: Distance to the found voxel or None if no voxel
                      is found within max_distance.
    """
    # 1. Normalize the direction vector so that the steps are uniform
    norm_direction = direction_vector / np.linalg.norm(direction_vector)

    # 2. Iterate along the ray
    current_distance = 0.0
    while current_distance < max_distance:
        # Compute the current point on the ray
        current_point = start_point + norm_direction * current_distance

        # 3. Define the search region around the current point
        # Round the coordinates to the nearest integers to get
        # the central voxel for the search
        center_voxel = np.round(current_point).astype(int)

        # Create the range of indices to search based on the radius
        search_radius = int(np.ceil(radius))
        min_coords = np.maximum(0, center_voxel - search_radius)
        max_coords = np.minimum(np.array(voxel_space.shape) - 1, center_voxel + search_radius)

        # 4. Check the voxels in the found region
        # Create a coordinate grid for checking
        z, y, x = np.ogrid[min_coords[0]:max_coords[0]+1,
                           min_coords[1]:max_coords[1]+1,
                           min_coords[2]:max_coords[2]+1]

        # Select the sub-array to check
        search_volume = voxel_space[z, y, x]

        # Find the coordinates of all 'True' voxels in this region
        true_voxels_indices = np.argwhere(search_volume)

        if true_voxels_indices.size > 0:
            # If 'True' voxels are found, check whether they are
            # within the spherical radius from current_point
            
            # Convert local indices to global ones
            global_indices = true_voxels_indices + min_coords
            
            # Compute the distance from current_point to each found voxel
            distances_to_true_voxels = np.linalg.norm(global_indices - current_point, axis=1)
            
            # If any voxel is inside the radius, return the distance
            if np.any(distances_to_true_voxels <= radius):
                return current_distance

        # 5. Move further along the ray
        current_distance += step_size

    # If we passed max_distance and found nothing
    return None

In [ ]:
angle_1 = {}
angle_2 = {}
angle_3 = {}
angle_4 = {}

angle_b1_1 = {}
angle_b1_2 = {}
angle_b2_1 = {}
angle_b2_2 = {}

bl1 = {}
bl2 = {}

bdl1 ={}
bdl2 = {}

lx = {}
ly = {}
lz = {}
G_4 = {}

for val in range(1, len(objects) + 1):
    if objects[val - 1] != None:
        try:
            pore = (halfthroats_img[objects[val - 1]] == val)

            fvec = flow_direction_vector[val]  # this gives much more accurate data
            centroid, axes = principal_axes_pca(pore)
            paxes = np.transpose(axes)

            angle_1[val] = abs(vectors_angle(fvec, paxes[0]))
            angle_2[val] = abs(vectors_angle(fvec, paxes[1]))

            border1 = (halfthroats_boundaries_img[objects[val - 1]] == halfthroat_boundaries[val][0])
            
            border2 = (halfthroats_boundaries_img[objects[val - 1]] == halfthroat_boundaries[val][1])
            
            centroid1, axes1 = principal_axes_pca(border1)
            paxes1 = np.transpose(axes1)
            
            centroid2, axes2 = principal_axes_pca(border2)
            paxes2 = np.transpose(axes2)

            bvec1 = centroid1 - centroid
            bvec2 = centroid2 - centroid

            bl1[val] = vector_length(bvec1) / (volume_dict[val]) ** (1 / 3)
            bl2[val] = vector_length(bvec2) / (volume_dict[val]) ** (1 / 3)

            bdl1[val] = find_distance_to_voxel(border1, centroid, centroid1 - centroid, radius = 0.75) / (volume_dict[val]) ** (1 / 3)
            bdl2[val] = find_distance_to_voxel(border2, centroid, centroid2 - centroid, radius = 0.75) / (volume_dict[val]) ** (1 / 3)


            angle_b1_1[val] = abs(vectors_angle(bvec1, paxes[0]))
            angle_b1_2[val] = abs(vectors_angle(bvec1, paxes[1]))
            angle_b2_1[val] = abs(vectors_angle(bvec2, paxes[0]))
            angle_b2_2[val] = abs(vectors_angle(bvec2, paxes[1]))

            
            lengths = compute_body_measures(pore, *paxes)
            lx[val] = lengths[0]
            ly[val] = lengths[1]
            lz[val] = lengths[2]
            G_4[val] = lengths[2] / lengths[0]
        except Exception as e:
            print(val, e)
    if val % 1000 == 0:
        print(f'current: {val}')

## Distance from the edge

In [ ]:
distance = {}

In [ ]:
for val in range(1, len(objects) + 1):
    if objects[val - 1] != None:
        hts = objects[val - 1]
        distance[val] = min([hts[0].start, raw_dimension - hts[0].stop,
                       hts[1].start, raw_dimension - hts[1].stop,
                       hts[2].start, raw_dimension - hts[2].stop,])

# Save everything to pandas

In [ ]:
import pandas as pd

In [ ]:
for i in angle_1.keys():
    if angle_1[i] > np.pi / 2:
        angle_1[i] = np.pi - angle_1[i]
for i in angle_2.keys():
    if angle_2[i] > np.pi / 2:
        angle_2[i] = np.pi - angle_2[i]

In [ ]:
features = {
    "g_x": g_x1,
    "g_y": g_y1,
    "g_z": g_z1,
    "pressure_x": pressure_x,
    "pressure_y": pressure_y,
    "pressure_z": pressure_z,
    "g": g,
    "distance_to_cube_border": distance,
    "G_vp": G_vp,
    "G_1": G_1,
    "G_2": G_2,
    "G_3": G_3,
    "G_4": G_4,
    "area": area_dict,
    "volume": volume_dict,
    "border1_G_4": border_1_G4,
    "border2_G_4": border_2_G4,
    "border1_relative_area": relative_area1,
    "border2_relative_area": relative_area2,
    "flow_angle_1": angle_1,
    "flow_angle_2": angle_2,
    "flow_angle_3": angle_3,
    "flow_angle_4": angle_4,
    "border1_angle_1": angle_b1_1,
    "border1_angle_2": angle_b1_2,
    "border2_angle_1": angle_b2_1,
    "border2_angle_2": angle_b2_2,
    "distance_to_border1_cm": bl1,
    "distance_to_border2_cm": bl2,
    "distance_to_border1_direct": bdl1,
    "distance_to_border2_direct": bdl2,
    "lenght_x": lx,
    "lenght_y": ly,
    "lenght_z": lz,
}

In [ ]:
df = pd.DataFrame(features)

In [ ]:
df.index.name = "halfthroat_id"

In [ ]:
df.to_csv(f"../datasets/{filename}.csv")